In [23]:
%pip install useful_rdkit_utils mols2grid


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import pandas as pd
from rdkit import Chem
import mols2grid
import useful_rdkit_utils as uru
from tqdm.auto import tqdm
from itertools import chain

In this notebook we'll analyze a set of marketed drugs from the ChEMBL database and find the most commonly occuring ring systems.  To do this, we'll follow these steps. 
1. Read the drugs as SMILES
2. Convert the SMILES to RDKit Molecules
3. Indentify the ring systems in the molecules
4. Collect the individual ring systems and count their frequencies

This analysis is similar to the one performed in Taylor, R. D., MacCoss, M., & Lawson, A. D. (2014). [Rings in drugs: Miniperspective](https://pubs.acs.org/doi/10.1021/jm4017625), Journal of Medicinal Chemistry, 57(14), 5845-5859.

Enable progress_apply in Pandas

In [25]:
tqdm.pandas()

### 1. Read drugs from ChEMBL as SMILES
Read the drugs from the ChEMBL database

In [26]:
import sys
sys.path.append("..")

from read_csv import read_csv

chembl_drugs_url = "https://raw.githubusercontent.com/PatWalters/datafiles/main/chembl_drugs.smi"
df = read_csv(chembl_drugs_url,sep=" ",names=["SMILES","Name"])

### 2. Convert the SMILES to RDKit Molecules
Add a molecule column to the dataframe

In [27]:
df['mol'] = df.SMILES.progress_apply(Chem.MolFromSmiles)

  0%|          | 0/1203 [00:00<?, ?it/s]

### 3. Indentify the ring systems in the molecules
Instantiate a RingSystemFinder object

In [28]:
ring_system_finder = uru.RingSystemFinder()

In [29]:
df['ring_systems'] = df.mol.progress_apply(ring_system_finder.find_ring_systems)

  0%|          | 0/1203 [00:00<?, ?it/s]

### 4. Collect the individual ring systems and count their frequencies
The ring_system column in **df** is a list of lists.  We need to flatten that list so we can count the number of times each ring system occurs.  The **chain** method in the itertools package provides a convenient way to do this. 

In [30]:
ring_list = chain(*df.ring_systems.values)
ring_list

The **chain** method used above returns an iterator.  We can use that iterator to create a Pandas series. 

In [31]:
ring_series = pd.Series(ring_list)
ring_series

0                           c1ccccc1
1                           c1ccncc1
2              O=C1CC(=O)NC(=O)[N-]1
3              O=C1C=CC(=O)c2ccccc21
4       O=c1[nH]c(=O)c2[nH]cnc2[nH]1
                    ...             
2453                        c1ccccc1
2454                        c1ccccc1
2455       O=c1[nH]c(=O)c2ccsc2[nH]1
2456                        c1ccnnc1
2457                        c1ccccc1
Length: 2458, dtype: str

Now that we have a Pandas series, we can use the value_counts method to count the occurences of the different ring systems.

In [32]:
ring_series.value_counts()

c1ccccc1                     911
c1ccncc1                      94
C1CNCCN1                      83
C1CCNCC1                      74
C1CC1                         48
                            ... 
c1ccc2c(c1)CCc1cccnc1C2        1
O=c1ccc2cnccc2[nH]1            1
O=c1cccn[nH]1                  1
O=c1[nH]c(=O)c2ccsc2[nH]1      1
c1ccnnc1                       1
Name: count, Length: 415, dtype: int64

In order to make the **value_counts** output easier to work with, we'll convert it into a dataframe. 

In [33]:
ring_df = pd.DataFrame(ring_series.value_counts()).reset_index()
ring_df.columns = ["SMILES","Count"]
ring_df

,SMILES,Count
0,c1ccccc1,911
1,c1ccncc1,94
2,C1CNCCN1,83
3,C1CCNCC1,74
4,C1CC1,48
...,...,...
410,c1ccc2c(c1)CCc1cccnc1C2,1
411,O=c1ccc2cnccc2[nH]1,1
412,O=c1cccn[nH]1,1
413,O=c1[nH]c(=O)c2ccsc2[nH]1,1


Now that we have our results in a dataframe, we can use mols2grid to display the chemical structures of the ring systems along with the associated counts. 

In [34]:
mols2grid.display(ring_df,smiles_col="SMILES",subset=["img","Count"],selection=False)

---

# Summary — Finding the Most Common Ring Systems in Drugs

**Goal:** Take 1203 marketed drugs from ChEMBL and find which ring systems appear most often.

## Setup
`tqdm.pandas()` — turns on progress bars for pandas operations (the `100%` bars you see).

## Step 1 — Load the drugs
Read a `.smi` file from ChEMBL into a dataframe `df` with two columns: `SMILES` (the drug as text) and `Name`. → 1203 rows.

## Step 2 — Text → Molecules
`Chem.MolFromSmiles` converts each SMILES string into a real RDKit molecule object, stored in a new `mol` column. Now the program can actually analyze the structures, not just read text.

## Step 3 — Extract the Rings
A `RingSystemFinder` scans each molecule and pulls out its ring systems (throwing away chains/side groups). Result: a `ring_systems` column where each drug has a list of its rings.

## Step 4 — Flatten and Count
- `chain(*...)` merges all the per-drug lists into one big pile of rings → **2458** rings total.
- `pd.Series` + `value_counts()` counts how many times each unique ring appears.
- `pd.DataFrame(...)` tidies the result into a table with columns `SMILES` and `Count` → **415** unique ring systems.

## Result (Top Rings)

| Ring | Count |
|---|---|
| `c1ccccc1` (benzene) | 911 |
| `c1ccncc1` (pyridine) | 94 |
| `C1CNCCN1` (piperazine) | 83 |
| `C1CCNCC1` (piperidine) | 74 |
| `C1CC1` (cyclopropane) | 48 |

## Final Step
`mols2grid.display` draws each ring's structure next to its count, so you see the picture instead of just the SMILES text.

## Takeaway
Benzene dominates — 911 of 2458 rings. A handful of common rings account for most of drug chemistry, matching the Taylor et al. (2014) *"Rings in Drugs"* study.